In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, TrainingArguments, Trainer
import torch

# model_checkpoint = "nguyenvulebinh/vi-mrc-large"
model_checkpoint = "/home/annv/Downloads/vi-mrc-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

# Load your dataset
data_files = {
    "train": "/home/annv/git/extractive-qa-mrc/data-bin/my-data/train.json",
    "validation": "/home/annv/git/extractive-qa-mrc/data-bin/my-data/validation.json"}
dataset = load_dataset("json", data_files=data_files)
dataset = dataset.with_format("torch")

# Preprocessing
def prepare_features(example):
    tokenized = tokenizer(
        example["question"],
        example["context"],
        truncation="only_second",
        max_length=512,
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )
    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        sequence_ids = tokenized.sequence_ids(i)

        sample_index = sample_mapping[i]
        answers = example["answers"][sample_index]
        if len(answers["answer_start"]) == 0:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        token_start_index = 0
        while sequence_ids[token_start_index] != 1:
            token_start_index += 1

        token_end_index = len(input_ids) - 1
        while sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                token_start_index += 1
            start_positions.append(token_start_index - 1)

            while token_end_index >= 0 and offsets[token_end_index][1] >= end_char:
                token_end_index -= 1
            end_positions.append(token_end_index + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized

tokenized_datasets = dataset.map(prepare_features, batched=True, remove_columns=dataset["train"].column_names)

# Training arguments
training_args = TrainingArguments(
    output_dir="./finetune/vi-mrc-base",
    # eval_strategy="steps",
    eval_steps=50,
    save_steps=100,
    logging_steps=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=50,
    save_total_limit=1,
    learning_rate=3e-5,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets.get("validation"),
    tokenizer=tokenizer,
)

trainer.train()


/tmp/ipykernel_3882712/2932156998.py:92: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/home/annv/.local/lib/python3.10/site-packages/transformers/utils/generic.py:271: FutureWarning: The input object of type 'Tensor' is an array-like implementing one of the corresponding protocols (`__array__`, `__array_interface__` or `__array_struct__`); but not a sequence (or 0-D). In the future, this object will be coerced as if it was first converted using `np.array(obj)`. To retain the old behaviour, you have to either modify the type 'Tensor', or assign to an empty array created with `np.empty(correct_shape, dtype=object)`.
  arr = np.array(obj)


Step,Training Loss


In [15]:
import torch
import torch.nn.functional as F

def answer_question(question, context, tokenizer, model):
    # Tokenize input with offset mapping
    inputs = tokenizer(question, context, return_tensors="pt", padding=True, truncation=True, return_offsets_mapping=True)
    offset_mapping = inputs.pop("offset_mapping")[0]  # Remove offset_mapping before passing to model

    # Run inference
    with torch.no_grad():
        outputs = model(**inputs)

    # Get logits and apply softmax
    start_logits = outputs.start_logits
    end_logits = outputs.end_logits

    start_probs = F.softmax(start_logits, dim=-1)
    end_probs = F.softmax(end_logits, dim=-1)

    # Get most probable start and end positions (token level)
    start_index = torch.argmax(start_probs).item()
    end_index = torch.argmax(end_probs).item()

    # Extract character positions from offset mapping
    start_char, _ = offset_mapping[start_index]
    _, end_char = offset_mapping[end_index]

    # Calculate confidence as geometric mean of start & end probabilities
    confidence = torch.sqrt(start_probs[0, start_index] * end_probs[0, end_index]).item()

    # Extract answer from the context
    answer = context[start_char:end_char]

    return answer, confidence, start_char, end_char

In [ ]:
# !pip install torch --upgrade
# !pip install transformers datasets accelerate evaluate


In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

# Load tokenizer and model from your finetuned directory
# model_path = "./vi-mrc-finetuned/checkpoint-20"
model_path = "./finetune/vi-mrc-base/checkpoint-"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForQuestionAnswering.from_pretrained(model_path)

# Put model in evaluation mode
model.eval()

# Your test input
# question = "tam nguyên phạm đôn lễ sinh năm nào" # Answer: năm 14
# context = "Phạm Đôn Lễ sinh năm 1457 tại làng Hải Triều (tục gọi là làng Hới), thuộc tổng Thanh Triều, phủ Long Hưng, huyện Ngự Thiên, tỉnh Hưng Yên (nay là thôn Hải Triều, xã Tân Lễ, huyện Hưng Hà, tỉnh Thái Bình) trong một gia đình nghèo khó, bố làm nghề chài lưới, mẹ bán quán nước cho khách qua đò. Khi ông còn rất nhỏ thì người bố qua đời, hai mẹ con đơn côi sống dựa vào hàng quán. Một lần Phạm Đôn Lễ bị lạc ở bờ sông Luộc, người mẹ đi tìm khắp nơi mà không được. Trong lúc lang thang vì lạc mẹ thì Phạm Đôn Lễ được một gia đình giàu có quê ở Thanh Hóa đưa lên thuyền về nhà nuôi dưỡng."
# context = "Tam Nguyên Phạm Đôn Lễ sinh năm 1457, mất năm 1531, là vị Tam Nguyên đầu tiên trong lịch sử khoa bảng Việt Nam. Ông đỗ Trạng nguyên năm 1481 dưới triều vua Lê Thánh Tông. Ông làm quan đến chức Thượng thư và từng được cử đi sứ nhà Minh năm 1488."

# question = "nhà thiên văn nào đã đề xuất thuyết nhật tâm" # Answer: Nicolaus Co
# context = "Nicolaus Copernicus đã phá vỡ quan niệm Trái đất nằm ở trung tâm của vũ trụ tồn tại suốt nhiều thế kỷ. Bằng những lập luận sắc bén trong thuyết nhật tâm, ông đề xuất rằng Trái đất và các hành tinh khác quay xung quanh Mặt trời."
# context = "Đối với Copernicus, lý thuyết nhật tâm của ông không hẳn là một bước ngoặt, bởi vì nó tạo ra nhiều vấn đề lớn cần phải giải quyết. Ví dụ, các vật thể nặng luôn được cho là rơi xuống mặt đất vì Trái đất là trung tâm của vũ trụ. Vậy tạo sao chúng lại rơi xuống đất trong một hệ thống lấy Mặt trời làm trung tâm?"
## NO ANSWER
# context = "Sau khi công trình cơ học thiên thể của Isaac Newton vào cuối thế kỷ 17 được công bố, sự chấp nhận thuyết nhật tâm lan truyền nhanh chóng ở các quốc gia ngoài Công giáo, và đến cuối thế kỷ 18, nó gần như được chấp nhận rộng rãi."
# context = "Hệ thống được ưa chuộng là hệ Ptolemy, trong đó Trái Đất nằm ở trung tâm vũ trụ và mọi thiên thể đều quay quanh nó. (Không nên lẫn lộn việc Cơ Đốc giáo ủng hộ thuyết địa tâm với ý tưởng về một Trái Đất phẳng, là cái chưa từng được Giáo hội ủng hộ.) Hệ Tycho đã sắp đặt ổn thỏa các vị trí của mô hình địa tâm, trong đó Mặt Trời quay quanh Trái Đất, trong khi các hành tinh quay quanh Mặt Trời giống như mô hình của Copernicus. Những nhà thiên văn học dòng Tên tại Roma ban đầu không đồng ý với hệ thống của Tycho; người nổi bật nhất là Clavius, ông đã bình luận rằng Tycho đã \"lẫn lộn mọi thứ trong thiên văn học, bởi vì ông muốn đặt Sao Hỏa thấp hơn Mặt Trời.\" (Fantoli, 2003, p. 109) Nhưng khi cuộc tranh cãi ngày càng phát triển và Giáo hội có quan điểm cứng rắn hơn về các ý tưởng của Copernicus sau năm 1616, dòng Tên quay sang ủng hộ việc giảng dạy ý tưởng của Tycho; sau năm 1633, việc sử dụng hệ thống này hầu như đã trở thành bắt buộc. Vì tội đã đề xuất thuyết nhật tâm, Galileo đã bị quản thúc tại gia trong nhiều năm."


# question = "tác dụng của xạ đen"
# context = "Về Đông y, cây xạ đen có vị hơi chát và đắng, tính hàn và có những tác dụng sau:\n- Điều trị bệnh viêm gan, xơ gan, hỗ trợ chữa gan nhiễm mỡ làm vàng da\n- Giải độc, tiêu viêm, mụn nhọt trên da\n- Ổn định huyết áp, hoạt huyết\n- Giúp giải tỏa căng thẳng, an thần, tăng sức đề kháng\n- Chữa khối u\n- Trị các bệnh xương khớp, cột sống\nTùy từng bài thuốc liều dùng xạ đen sẽ tương ứng, tuy nhiên tối đa chỉ nên dùng xạ đen khoảng 70g/ngày và cần tham khảo ý kiến thầy thuốc để được tư vấn liều dùng phù hợp."

# context = "Dưới đây là cách phân biệt cây xạ đen với những cây khác cùng họ:\n- Cây xạ đen: Cây tươi có lá dày và màu tím xanh, thân cây đậm màu. Sau khi phơi khô, lá cây bị nát nhưng không giòn, có mùi thơm nhẹ, thân cây chuyển sang màu đen và có mùi thơm.\n- Cây xạ vàng: Cây tương có lá mỏng và màu xanh, mép lá không có răng cưa. Sau khi phơi khô, lá cây rất dễ bị nát và giòn, thân cây chuyển sang màu trắng và không có mùi thơm.\nNgoài các cây cùng họ, xạ đen cũng bị nhầm lẫn với cây chùm rụm, cây dót và cây xạ đen Hòa Bình. Tuy nhiên, về thành phần khi được nghiên cứu lại thấy rất khác nhau, đặc biệt là khả năng ức chế tế bào ung thư phổi và gan.",
# question = "cách phân biệt cây xạ đen",

# context = "Nguyễn Trãi (chữ Hán: 阮廌; 1380 – 19 tháng 9 năm 1442), hiệu là Ức Trai (抑齋), là một nhà chính trị, nhà văn, nhà văn hóa lớn của dân tộc Việt Nam. Ông đã tham gia tích cực cuộc Khởi nghĩa Lam Sơn do Lê Lợi lãnh đạo chống lại sự xâm lược của nhà Minh (Trung Quốc) với Đại Việt. Khi cuộc khởi nghĩa thành công vào năm 1428, Nguyễn Trãi trở thành một trong những khai quốc công thần của triều đại quân chủ nhà Hậu Lê trong Lịch sử Việt Nam.[2]"
# question = "Nguyễn Trãi là ai"

# context = "Nguyễn Hiền ( chữ Hán : 阮賢, 11 tháng 3, 1234 - 05 tháng 9, 1256 ) là trạng nguyên trẻ nhất trong lịch sử khoa cử Việt Nam, khi mới mười ba tuổi."
# question = "ông trạng nguyễn hiền là ai"
# question = "ai là trạng nguyên khi mới 13 tuổi"

# question = "Trần Hữu Lượng là ai"
# context = """Trần Hữu Lượng (chữ Hán: 陳友諒; 1316– 3 tháng 10 năm 1363) là một thủ lĩnh quân phiệt thời "Nguyên mạt Minh sơ" trong lịch sử Trung Quốc, là người Miện Dương, Hồ Bắc, ông là người sáng lập nước Đại Hán Trung Quốc Tự Xưng Hoàng Đế Đại Hán ở Trung Quốc."""

# question = "mozart là ai"
# context = "Wolfgang Amadeus Mozart (tiếng Đức: [ˈvɔlfɡaŋ amaˈdeus ˈmoːtsart]; tên đầy đủ là Johannes Chrysostomus Wolfgangus Theophilus Mozart[cần dẫn nguồn] (27 tháng 1 năm 1756 – 5 tháng 12 năm 1791) là nhà soạn nhạc người Áo. Ông là một trong những nhà soạn nhạc nổi tiếng, quan trọng và có nhiều ảnh hưởng nhất trong thể loại nhạc cổ điển châu Âu."

question = "april là tháng mấy"
context = "Như chúng ta đã thấy trong danh sách ở trên, April tương ứng với tháng Tư trong tiếng Anh và thường được viết tắt là Apr."

answer, confidence, start_char, end_char = answer_question(question, context, tokenizer, model)
print(f"answer: {answer}")
print(f"confidence: {confidence:.4f}")
print(f"[start:end]: [{start_char}:{end_char}]")


answer: 
confidence: 0.8108
[start:end]: [65:0]
